# 2주차 실습 4 — 한국어 형태소 분석의 난제

**이 노트북의 새 개념**: 교착어의 어절 분해, 띄어쓰기 교정, 복합명사 분해를 kiwipiepy 로 확인한다.

KoNLPy 0.6.0 은 2022년 1월 이후 갱신이 없고 Java 가 필요해 이 수업에서는 비교용으로만 언급한다.

In [ ]:
%pip install -q kiwipiepy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 28.0 MB/s eta 0:00:00


In [ ]:
import sys, kiwipiepy
from kiwipiepy import Kiwi
print("Python", sys.version.split()[0], "| kiwipiepy", kiwipiepy.__version__)
kiwi = Kiwi()

Python 3.13.15 | kiwipiepy 0.23.2


## 어절 = 어간 + 문법 형태소

In [ ]:
for s in ["먹었습니다", "먹었지만", "먹겠어요"]:
    print(f"{s:<8}", [(t.form, t.tag) for t in kiwi.tokenize(s)])

먹었습니다    [('먹', 'VV'), ('었', 'EP'), ('습니다', 'EF')]
먹었지만     [('먹', 'VV'), ('었', 'EP'), ('지만', 'EC')]
먹겠어요     [('먹', 'VV'), ('겠', 'EP'), ('어요', 'EF')]


## 띄어쓰기 교정
규범과 실제 사용이 자주 어긋나므로, 분석 전에 띄어쓰기를 바로잡는 단계가 따로 있다.

In [ ]:
for s in ["아버지가방에들어가신다", "텍스트마이닝은재미있다"]:
    print(f"{s} → {kiwi.space(s)}")

아버지가방에들어가신다 → 아버지가 방에 들어가신다
텍스트마이닝은재미있다 → 텍스트 마 이닝은 재미있다


## 복합명사 — 사용자 사전으로 분해 단위를 바꾼다
사전 없이 분석한 결과와, 사용자 단어를 등록한 뒤의 결과를 비교한다.

In [ ]:
s = "자연어처리연구실에서 형태소분석기를 비교했다"
print("기본   :", [t.form for t in kiwi.tokenize(s)])
kiwi.add_user_word("자연어처리", "NNP")
kiwi.add_user_word("형태소분석기", "NNP")
print("사전 추가:", [t.form for t in kiwi.tokenize(s)])

기본   : ['자연어 처리', '연구실', '에서', '형태소', '분석기', '를', '비교', '하', '었', '다']
사전 추가: ['자연어처리', '연구실', '에서', '형태소분석기', '를', '비교', '하', '었', '다']


## 말뭉치에서 명사만 뽑기

In [ ]:
# # 말뭉치 올리기 — 1주차와 같은 파일을 쓴다(빈 줄로 구분된 문단 하나를 문서 하나로 본다)
# CORPUS_PATH = "/content/corpus.txt"      # Colab 밖에서 실행할 때는 이 경로를 직접 바꾼다
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     CORPUS_PATH = "/content/" + next(iter(uploaded))
# except ImportError:
#     pass

# with open(CORPUS_PATH, encoding="utf-8") as f:
#     raw = f.read()
# docs = [d.strip() for d in raw.split("\n\n") if len(d.strip()) > 20]   # 문단 = 문서
# print(f"문서 {len(docs):,}개, 예시: {docs[0][:60]}...")

In [ ]:
# 말뭉치 올리기 — 한 줄을 하나의 문서로 사용
CORPUS_PATH = "/content/corpus.txt"

try:
    from google.colab import files
    uploaded = files.upload()
    CORPUS_PATH = "/content/" + next(iter(uploaded))
except ImportError:
    pass

docs = []

with open(CORPUS_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        parts = line.split("\t", 1)

        if len(parts) == 2:
            label, text = parts
            text = text.strip()

            if text:
                docs.append(text)

print(f"문서 {len(docs):,}개")
print("예시:", docs[0])

Saving corpus_100k_tokens.txt to corpus_100k_tokens.txt
문서 100,000개
예시: 노래가 너무 적음


In [ ]:
from collections import Counter
nouns = Counter(t.form for d in docs[:500] for t in kiwi.tokenize(d) if t.tag.startswith("NN"))
print("상위 명사:", nouns.most_common(15))

상위 명사: [('게임', 215), ('거', 92), ('것', 91), ('수', 39), ('시간', 29), ('때', 27), ('플레이', 26), ('사람', 25), ('개', 24), ('스토리', 22), ('재미', 21), ('분', 19), ('생각', 19), ('처음', 18), ('데', 17)]


In [ ]:
# 과제: 형태소 분석 오류 사례 5개 분석
# 분류: 띄어쓰기 / 복합명사 / 신조어 / 기타

cases = [
    {
        "sentence": "서든어택 표절함 ㅋㅋㅋㅋㅋ",
        "category": "복합명사",
        "user_word": "서든어택",
        "tag": "NNP"
    },
    {
        "sentence": "한글패치 꼭좀 나왔으면 좋겠습니다",
        "category": "복합명사",
        "user_word": "한글패치",
        "tag": "NNG"
    },
    {
        "sentence": "개꿀잼 게임",
        "category": "신조어",
        "user_word": "개꿀잼",
        "tag": "NNG"
    },
    {
        "sentence": "갓겜 세일할때 꼭 사셈",
        "category": "신조어",
        "user_word": "갓겜",
        "tag": "NNG"
    },
    {
        "sentence": "텍스트마이닝은재미있다",
        "category": "띄어쓰기",
        "user_word": None,
        "tag": None
    }
]

for i, case in enumerate(cases, 1):
    s = case["sentence"]

    print("=" * 70)
    print(f"Case {i}")
    print("문장:", s)
    print("오류 유형:", case["category"])

    # 기본 분석
    before = [(t.form, t.tag) for t in kiwi.tokenize(s)]
    print("기본 분석:", before)

    # 띄어쓰기 문제
    if case["category"] == "띄어쓰기":
        corrected = kiwi.space(s)
        print("띄어쓰기 교정:", corrected)
        print("교정 후:", [(t.form, t.tag) for t in kiwi.tokenize(corrected)])

    # 사용자 사전으로 수정 가능한 경우
    elif case["user_word"] is not None:
        kiwi.add_user_word(case["user_word"], case["tag"])
        after = [(t.form, t.tag) for t in kiwi.tokenize(s)]
        print(f"사용자 사전 추가: {case['user_word']}")
        print("사전 추가 후:", after)

Case 1
문장: 서든어택 표절함 ㅋㅋㅋㅋㅋ
오류 유형: 복합명사
기본 분석: [('서든어택', 'NNP'), ('표절', 'NNG'), ('하', 'XSV'), ('ᆷ', 'EF'), ('ㅋㅋㅋㅋㅋ', 'SW')]
사용자 사전 추가: 서든어택
사전 추가 후: [('서든어택', 'NNP'), ('표절', 'NNG'), ('하', 'XSV'), ('ᆷ', 'EF'), ('ㅋㅋㅋㅋㅋ', 'SW')]
Case 2
문장: 한글패치 꼭좀 나왔으면 좋겠습니다
오류 유형: 복합명사
기본 분석: [('한글', 'NNG'), ('패치', 'NNG'), ('꼭', 'MAG'), ('좀', 'MAG'), ('나오', 'VV'), ('었', 'EP'), ('으면', 'EC'), ('좋', 'VA'), ('겠', 'EP'), ('습니다', 'EF')]
사용자 사전 추가: 한글패치
사전 추가 후: [('한글패치', 'NNG'), ('꼭', 'MAG'), ('좀', 'MAG'), ('나오', 'VV'), ('었', 'EP'), ('으면', 'EC'), ('좋', 'VA'), ('겠', 'EP'), ('습니다', 'EF')]
Case 3
문장: 개꿀잼 게임
오류 유형: 신조어
기본 분석: [('개꿀잼', 'NNG'), ('게임', 'NNG')]
사용자 사전 추가: 개꿀잼
사전 추가 후: [('개꿀잼', 'NNG'), ('게임', 'NNG')]
Case 4
문장: 갓겜 세일할때 꼭 사셈
오류 유형: 신조어
기본 분석: [('갓겜', 'NNP'), ('세일', 'NNG'), ('하', 'XSV'), ('ᆯ', 'ETM'), ('때', 'NNG'), ('꼭', 'MAG'), ('사셈', 'NNG')]
사용자 사전 추가: 갓겜
사전 추가 후: [('갓겜', 'NNG'), ('세일', 'NNG'), ('하', 'XSV'), ('ᆯ', 'ETM'), ('때', 'NNG'), ('꼭', 'MAG'), ('사셈', 'NNG')]
Case 5
문장: 텍스트마이닝은재미있다
오류 유형: 띄어쓰기
기본 분석

### Error Analysis

Five error-prone Korean examples were examined using Kiwi.

- **띄어쓰기**: Missing spaces can cause incorrect segmentation.  
  `kiwi.space()` can improve the input before morphological analysis.

- **복합명사**: Proper nouns or compound nouns such as `서든어택` and `한글패치`
  may be split into smaller units. Adding them to the user dictionary can preserve
  them as one token.

- **신조어**: Internet expressions such as `개꿀잼` and `갓겜` may not be analyzed
  as intended because they are informal or newly created words. A user dictionary
  can improve their tokenization.

Therefore, spacing correction and user-dictionary registration are useful ways
to reduce Korean morphological-analysis errors.

## 직접 해 보기
1. 분석이 틀린 문장 3개를 찾아 원인(띄어쓰기·복합명사·신조어)을 분류해 보자.
2. 사용자 사전에 등록한 단어가 TF-IDF 상위어(`bow_tfidf.ipynb`)를 어떻게 바꾸는가?
3. KoNLPy 가 설치된 환경이 있다면 Okt 결과와 비교해 보자(Colab 에서는 Java 설치가 필요하다).